# 01 · Entendimento do negócio e dos dados (v2)
**CRISP-DM:** Entendimento do Negócio + Entendimento dos Dados

## 1. Entendimento do negócio
**Problema.** Quando um incidente é aberto, o operador precisa saber se as APIs (recursos) envolvidas estão operando dentro de um **regime normal** e, se não estão, **qual indicador saiu do padrão**: erros de negócio (4xx), erros de sistema (5xx) ou latência. O comportamento normal de uma API muda com o horário, o dia e os processamentos em lote, e por isso não basta um limite fixo.

**Decisão apoiada.** *O recurso estava em regime normal no momento do registro do incidente? Se não, qual indicador foi impactado e desde quando?* Como informação complementar: o impacto é **generalizado** ou concentrado em um **consumidor específico** capaz de alterar o resultado agregado?

**Objetivo de mineração.** Para cada recurso: (1) caracterizar seus regimes normais a partir dos Access Logs agregados; (2) no instante do registro de um incidente, classificar o período imediatamente anterior como **Normal** ou **Residual**; (3) apontar o **indicador responsável** e estimar o **início** do desvio.

**Questões de pesquisa.**
- **RQ1:** Access Logs agregados permitem caracterizar os regimes normais de cada recurso e decidir, no instante do registro, se o recurso está em regime normal?
- **RQ2:** Quando o recurso não está normal, o método identifica corretamente o indicador responsável (4xx, 5xx, latência) e o início do desvio?
- **RQ3:** Qual o menor período antes do registro (15, 30 ou 60 min) que já permite essa decisão, e como o atraso entre a degradação e o registro afeta o resultado?

**Critérios de sucesso** (definidos antes da modelagem):

| Critério | Meta |
|---|---|
| Consultas de controle (sem falha) com saída Normal | ≥ 95% |
| Sensibilidade no registro, faixa de 15 min, intensidade média ou alta | ≥ 70% |
| Acurácia do indicador responsável (casos detectados) | ≥ 90% |
| Início real dentro da zona estimada | ≥ 70% |
| Sensibilidade do método proposto ≥ baselines (McNemar) | sem perda significativa |

In [1]:
%run ./00_funcoes.ipynb

00_funcoes (v2) carregado · /home/wsl/gh/mestrado/claude9/v2


## 2. Entendimento dos dados
**Cada linha da base é um agregado por minuto × recurso × código HTTP × cliente**: `QT_TTL_TRAN` é o total de execuções naquele minuto e `QT_TMP_MEDI_EXEA`, o tempo médio de execução delas. Não há registros por requisição; por isso, percentis de latência são aproximados a partir dessas médias por minuto (notebook 02). `TS_EVT` está em UTC e é convertido para o horário de Brasília. `CD_CLI` vem anonimizado (SHA-256 com semente).

In [2]:
bruto = carregar()
print(f"linhas: {len(bruto):,}")
print("período (Brasília):", bruto["ts"].min(), "→", bruto["ts"].max())
print("recursos:", bruto["recurso"].nunique(), "· clientes:", bruto["cliente"].nunique())
print("status:", sorted(bruto["status"].unique()))
print("nulos:", int(bruto.isna().sum().sum()), "· qtd ≤ 0:", int((bruto["qtd"] <= 0).sum()), "· tmp < 0:", int((bruto["tmp"] < 0).sum()))

linhas: 2,300,414
período (Brasília): 2026-07-31 21:00:00 → 2026-09-08 20:59:00
recursos: 12 · clientes: 1825
status: [np.int64(200), np.int64(201), np.int64(400), np.int64(401), np.int64(403), np.int64(412), np.int64(416), np.int64(422), np.int64(423), np.int64(429), np.int64(499), np.int64(500), np.int64(502), np.int64(503), np.int64(504)]
nulos: 0 · qtd ≤ 0: 0 · tmp < 0: 0


### Códigos dos recursos
No artigo, os recursos são identificados por **E01–E12**, em ordem decrescente de volume. O mapeamento fica só nos resultados.

In [3]:
vol = bruto.groupby("recurso")["qtd"].sum().sort_values(ascending=False)
mapa = {int(r): f"E{i + 1:02d}" for i, r in enumerate(vol.index)}
arquivo_codigos().write_text(json.dumps(mapa, indent=2))
pd.Series(mapa, name="código")

18169917    E01
18917906    E02
18917920    E03
18344302    E04
18166630    E05
18917921    E06
18170306    E07
18139260    E08
18168196    E09
17834257    E10
18917905    E11
14553746    E12
Name: código, dtype: str

### Chaves repetidas

In [4]:
chave = ["ts", "recurso", "status", "cliente"]
rep = bruto.duplicated(chave, keep=False)
qualidade = dict(
    linhas=len(bruto), linhas_em_chave_repetida=int(rep.sum()), linhas_excedentes=int(bruto.duplicated(chave).sum()),
    frac_volume_em_chave_repetida=float(bruto.loc[rep, "qtd"].sum() / bruto["qtd"].sum()),
    iguais_nas_metricas=int(bruto[rep].duplicated(chave + ["qtd", "tmp"]).sum()))
qualidade

{'linhas': 2300414,
 'linhas_em_chave_repetida': 25285,
 'linhas_excedentes': 17265,
 'frac_volume_em_chave_repetida': 0.009521663975675615,
 'iguais_nas_metricas': 13341}

### Cobertura por recurso

In [5]:
cob = bruto.groupby("recurso").agg(inicio=("ts", "min"), fim=("ts", "max"), minutos=("ts", "nunique"),
                                   clientes=("cliente", "nunique"), volume=("qtd", "sum"))
cob["cobertura_min"] = cob["minutos"] / (((cob["fim"] - cob["inicio"]) / pd.Timedelta("1min")) + 1)
cob["part_volume"] = cob["volume"] / cob["volume"].sum()
cob["transacoes_por_15min_mediana"] = bruto.assign(j=bruto["ts"].dt.floor("15min")).groupby(["recurso", "j"])["qtd"].sum().groupby("recurso").median()
cob.index = cob.index.map(mapa)
cob = cob.sort_index()
cob

,inicio,fim,minutos,clientes,volume,cobertura_min,part_volume,transacoes_por_15min_mediana
recurso,,,,,,,,
E01,2026-07-31 21:00:00,2026-09-08 20:59:00,56094,36,12939466,0.998825,0.360549,1198.0
E02,2026-07-31 21:00:00,2026-09-03 07:11:00,47687,524,7195897,0.990755,0.200508,1876.0
E03,2026-07-31 21:00:00,2026-09-03 07:11:00,47071,221,2057817,0.977956,0.057340,502.0
E04,2026-07-31 21:03:00,2026-09-08 20:59:00,44109,20,1762363,0.785459,0.049107,208.0
E05,2026-07-31 21:00:00,2026-09-08 20:59:00,52778,18,1723640,0.939779,0.048028,104.0
E06,2026-07-31 21:00:00,2026-09-03 07:11:00,47679,735,1701480,0.990588,0.047410,313.0
E07,2026-07-31 21:00:00,2026-09-08 20:58:00,46006,36,1675513,0.819210,0.046687,146.0
E08,2026-07-31 21:00:00,2026-09-08 20:59:00,56010,38,1641156,0.997329,0.045730,314.0
E09,2026-07-31 21:03:00,2026-09-08 20:59:00,49132,27,1485713,0.874904,0.041398,142.0


### Proporção por família HTTP e latência

In [6]:
fam = bruto.groupby(bruto["status"] // 100)["qtd"].sum()
display((fam / fam.sum()).round(4))
desc = bruto[["qtd", "tmp"]].describe(percentiles=[0.5, 0.9, 0.99]).T
extremos = dict(latencia_acima_60s=int((bruto["tmp"] > 60_000).sum()), latencia_zero=int((bruto["tmp"] == 0).sum()))
display(desc)
extremos

status
2    0.7931
4    0.2040
5    0.0028
Name: qtd, dtype: float64

,count,mean,std,min,50%,90%,99%,max
qtd,2300414.0,15.600785,75.136563,1.0,3.0,22.0,248.0,2165.0
tmp,2300414.0,77.476336,2858.080189,0.0,39.0,83.0,308.0,1015962.0


{'latencia_acima_60s': 33, 'latencia_zero': 1406}

## 3. Regras de validade
1. Conversão de `TS_EVT` de UTC para America/Sao_Paulo antes de qualquer corte temporal.
2. Consolidação das chaves repetidas (soma de volume, média de latência ponderada).
3. Latências extremas mantidas (estatísticas robustas).
4. Cada recurso é avaliado apenas enquanto há dados (quatro recursos param em 03/09 às 07:11).

In [7]:
qualidade.update(extremos, periodo_inicio=bruto["ts"].min(), periodo_fim=bruto["ts"].max(),
                 recursos=int(bruto["recurso"].nunique()), clientes=int(bruto["cliente"].nunique()),
                 familias={int(k): float(v) for k, v in (fam / fam.sum()).items()})
salvar_json(qualidade, "01_qualidade.json")
cob.to_csv(RESULTADOS / "01_cobertura_recursos.csv")